# Haystack Pipeline — Stateless DAG Components with Named Port Wiring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/84-haystack-pipeline/haystack_pipeline_workbook.ipynb)

**Haystack 2.x** organises NLP work as a DAG of **Components** connected by named ports.
Unlike LangGraph — where nodes share a typed state dict — Haystack components are fully decoupled:
the only way data moves between them is through explicit port wiring.

**What you'll learn:**
- How Haystack 2.x differs from LangGraph in data flow and statefulness
- How to build a BM25-retrieval + LLM-generation pipeline
- How `pipeline.connect()` wires named output ports to named input ports
- The trade-offs between stateless pipelines and stateful graphs

## Workshop Roadmap

| # | Section | Exercise |
|---|---------|----------|
| 1 | Haystack 2.x Architecture | Understand DAG + ports |
| 2 | Building the Pipeline | Wire retriever → builder → generator |
| 3 | Running the Pipeline | Query the RAG pipeline |
| 4 | Inspecting the Pipeline | Access intermediate outputs |
| 5 | Stateless vs Stateful | Trade-off analysis |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "haystack-ai", "openai", "python-dotenv"],
        check=True,
    )
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print("API key loaded." if key else "WARNING: OPENAI_API_KEY not set.")

## Part 1 — Haystack 2.x Architecture

A Haystack `Pipeline` is a **directed acyclic graph (DAG)** of `Component` objects.
Components communicate **only through explicitly wired named ports** — there is no shared state dict.

```
query ──→ [InMemoryBM25Retriever] ──documents──→ [PromptBuilder] ──prompt──→ [OpenAIGenerator]
    └──────────────────────────────────────────────────────────────────────────────────→ query
```

The call `pipeline.connect("retriever.documents", "prompt_builder.documents")` wires the
`documents` **output port** of the retriever to the `documents` **input port** of the prompt builder.

**Key contrast with LangGraph:**

| Dimension | LangGraph | Haystack 2.x |
|-----------|-----------|---------------|
| Data flow | Shared typed state dict | Named port wiring |
| Statefulness | Built-in (MemorySaver, etc.) | Stateless by default |
| Component reuse | Nodes are project-specific | Components are reusable objects |
| Loop support | First-class (conditional edges) | DAG only — no cycles |

BM25 is a classic keyword-scoring algorithm — no embeddings needed, fast for small corpora.

In [ ]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

DOCUMENTS = [
    Document(content="LangGraph is a library for building stateful multi-actor LLM applications."),
    Document(content="Haystack is an open-source NLP framework with a component-based pipeline architecture."),
    Document(content="Redis is an in-memory data structure server for caching and pub/sub."),
    Document(content="ChromaDB is an open-source vector database for storing and querying embeddings."),
    Document(content="RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation."),
    Document(content="The Google ADK simplifies building AI agents powered by Gemini models."),
]

store = InMemoryDocumentStore()
store.write_documents(DOCUMENTS)
print(f"Document store initialized with {store.count_documents()} documents.")

## Part 2 — Building the Pipeline

Three steps:
1. **`add_component(name, instance)`** — register a component with a name
2. **`connect("a.output_port", "b.input_port")`** — wire data flow between components
3. Haystack validates connectivity before you can run — no silent misconfiguration

The `PromptBuilder` uses Jinja2 templating. Variables in `{{ }}` are filled from incoming ports.
The `query` variable comes from the run-time input dict; `documents` comes from the retriever output.

In [ ]:
from haystack import Pipeline
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator

PROMPT_TEMPLATE = '''
Answer the question using only the provided documents.
If the documents don't contain the answer, say so.

Documents:
{% for doc in documents %}
- {{ doc.content }}
{% endfor %}

Question: {{ query }}
Answer:
'''.strip()

retriever     = InMemoryBM25Retriever(document_store=store, top_k=3)
prompt_builder = PromptBuilder(template=PROMPT_TEMPLATE)
generator     = OpenAIGenerator(model="gpt-5.4-nano")

pipeline = Pipeline()
pipeline.add_component("retriever",      retriever)
pipeline.add_component("prompt_builder", prompt_builder)
pipeline.add_component("generator",      generator)

# Wire named ports: retriever.documents → prompt_builder.documents
pipeline.connect("retriever.documents",   "prompt_builder.documents")
# Wire named ports: prompt_builder.prompt → generator.prompt
pipeline.connect("prompt_builder.prompt", "generator.prompt")

print("Pipeline built. Components:")
for name in ["retriever", "prompt_builder", "generator"]:
    print(f"  {name}")

## Part 3 — Running the Pipeline

`pipeline.run()` takes a **dict keyed by component name**.
Each component only receives the keys it declares as inputs.

Notice that `query` goes to **both** `retriever` and `prompt_builder`:
- `retriever` uses it for BM25 keyword scoring
- `prompt_builder` uses it to fill the `{{ query }}` Jinja variable

This explicit passing is by design — no hidden global state.

In [ ]:
def run_query(query: str) -> str:
    result = pipeline.run({
        "retriever":      {"query": query},
        "prompt_builder": {"query": query},
    })
    replies = result["generator"]["replies"]
    return replies[0] if replies else "(no answer)"


# Test
query = "What is Haystack?"
print(f"Q: {query}")
print(f"A: {run_query(query)}")

## Part 4 — Inspecting the Pipeline

The full result dict from `pipeline.run()` contains one key per component.
You can access intermediate outputs by indexing `result["component_name"]`.

This makes debugging straightforward: check what the retriever returned,
what the prompt looked like before it hit the model, and what the model replied.

In LangGraph, you'd check `state` between nodes. In Haystack, you read the result dict.

In [ ]:
# Run and inspect the full result structure.

query = "What is RAG?"

full_result = pipeline.run({
    "retriever":      {"query": query},
    "prompt_builder": {"query": query},
})

print("=== Top-level result keys ===")
print(list(full_result.keys()))

print("\n=== Generator replies ===")
for i, reply in enumerate(full_result["generator"]["replies"]):
    print(f"Reply {i + 1}: {reply}")

print("\n=== Running more queries ===")
for q in ["How does LangGraph work?", "What is ChromaDB?"]:
    print(f"\nQ: {q}")
    print(f"A: {run_query(q)}")

## Part 5 — Stateless vs Stateful

| Dimension | Haystack (stateless) | LangGraph (stateful) |
|-----------|----------------------|----------------------|
| Memory across runs | None by default | Built-in (MemorySaver, etc.) |
| Component coupling | Decoupled via ports | Nodes share state dict |
| Reusability | High — components are generic | Lower — nodes are graph-specific |
| Flow control | DAG only | Conditional edges, loops |
| Debugging | Inspect result dict | Trace state at each node |

**When to use Haystack:**
- Document ingestion, chunking, embedding, retrieval pipelines
- Batch NLP workflows (classification, summarisation, extraction)
- When you want reusable, swappable components

**When to use LangGraph:**
- Conversational agents that need memory across turns
- Complex branching logic (human-in-the-loop, retry loops)
- Production systems needing full audit trails

In [ ]:
# Statelessness demonstration: two identical runs produce identical results.
# The pipeline has no memory of Run 1 during Run 2.

query = "What is Redis?"

print("Run 1:")
print(run_query(query))

print("\nRun 2 (identical — pipeline has no state between runs):")
print(run_query(query))

# In LangGraph you'd need explicit state management to get this idempotent behaviour.
# In Haystack, statelessness is the default — the trade-off is no cross-run memory.
print("\nStatelessness confirmed: pipeline has no memory of Run 1 during Run 2.")

## Exercises

**Exercise 1 — Add more documents**

Add 3 new `Document` objects to the `store` using `store.write_documents()`.
Then run queries that require the new content and verify the pipeline returns relevant answers.

**Exercise 2 — Change top_k**

The retriever currently returns `top_k=3` documents.
Rebuild the pipeline with `top_k=1` and run the same queries.
Observe how answer quality changes when the model has fewer documents to reason over.

In [ ]:
# Exercise 1 — Add more documents and verify retrieval.

from haystack import Document

new_docs = [
    Document(content="FastAPI is a modern, high-performance Python web framework."),
    Document(content="Docker containers package applications with all their dependencies."),
    Document(content="PostgreSQL is a powerful open-source relational database system."),
]
store.write_documents(new_docs)
print(f"Document store now has {store.count_documents()} documents.")

# Test with new content
new_queries = ["What is FastAPI?", "How does Docker work?"]
for q in new_queries:
    print(f"\nQ: {q}")
    print(f"A: {run_query(q)}")

---

**Next:** [85 — Agno Agent](../85-agno-agent/agno_agent_workbook.ipynb) — minimal boilerplate agents with Agno.